# SOC Threat Detection & MITRE ATT&CK Mapping

**Cybersecurity Project 3 — Lesson 03, Demo 02 enhancement**  
**Environment:** Google Colab | **Language:** Python | **Data:** synthetic SIEM events only

This notebook follows the lesson's idea of integrating SIEM logs with MITRE ATT&CK technique data and adds **rule-based detections, severity scoring, interactive visualizations, a security report, and GitHub-ready exports**.

**Important:** A detection is a *possible indicator* requiring analyst review, not proof of a real compromise. No real user logs, passwords, or live systems are involved. This is **not a trained AI model**.

**How to run:** Select **Runtime → Run all**. If Colab prompts you, approve the trusted notebook execution. The last cell downloads a ZIP of the project outputs.

## 1. Install libraries

The original lesson lists `stix2`, `pandas`, `requests`, and `pycti`. This enhanced notebook uses `stix2`, `pandas`, and `requests`, plus `plotly` for its dashboard. `pycti` is not required because we are not connecting to OpenCTI.

In [1]:
%pip -q install stix2 pandas requests plotly

## 2. Load MITRE ATT&CK Enterprise techniques

This step tries to load the official MITRE STIX 2.0 JSON dataset used in the lesson. The dataset is large, so the first run can take some time. If the download fails, the notebook uses a small, labelled offline reference list of the techniques used in the sample — you can still finish and test the project.

References: [MITRE ATT&CK](https://attack.mitre.org/) and [MITRE CTI repository](https://github.com/mitre/cti).

In [2]:
import re
import json
import datetime as dt
from pathlib import Path

import pandas as pd
import requests
import plotly.express as px
from IPython.display import display, HTML

PROJECT = Path('/content/SOC_MITRE_Project')
PROJECT.mkdir(parents=True, exist_ok=True)

# Verified technique identifiers used by this synthetic scenario.
FALLBACK_TECHNIQUES = {
    'T1059.001': 'Command and Scripting Interpreter: PowerShell',
    'T1003.001': 'OS Credential Dumping: LSASS Memory',
    'T1110': 'Brute Force',
    'T1566.001': 'Phishing: Spearphishing Attachment',
    'T1566.002': 'Phishing: Spearphishing Link',
    'T1071.001': 'Application Layer Protocol: Web Protocols',
    'T1021.001': 'Remote Services: Remote Desktop Protocol',
    'T1136.001': 'Create Account: Local Account',
    'T1098': 'Account Manipulation',
    'T1059.004': 'Command and Scripting Interpreter: Unix Shell',
}

MITRE_STIX_URL = 'https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json'
technique_catalog = dict(FALLBACK_TECHNIQUES)
data_source = 'Bundled MITRE technique reference (offline fallback)'

try:
    response = requests.get(MITRE_STIX_URL, timeout=35)
    response.raise_for_status()
    objects = response.json().get('objects', [])
    live_catalog = {}
    for obj in objects:
        if obj.get('type') != 'attack-pattern' or obj.get('revoked') or obj.get('x_mitre_deprecated'):
            continue
        for reference in obj.get('external_references', []):
            tid = reference.get('external_id', '')
            if reference.get('source_name') == 'mitre-attack' and re.fullmatch(r'T\d{4}(?:\.\d{3})?', tid):
                live_catalog[tid] = obj.get('name', '')
                break
    if live_catalog:
        technique_catalog.update(live_catalog)
        data_source = 'Live MITRE Enterprise ATT&CK STIX dataset'
        print(f'Loaded {len(live_catalog)} ATT&CK techniques from official MITRE STIX data.')
    else:
        print('Could not extract technique IDs; using the bundled reference list.')
except (requests.RequestException, ValueError, MemoryError) as ex:
    print('Live MITRE download unavailable; using built-in reference list.')
    print('Reason:', str(ex)[:180])

print('Catalog:', data_source)
display(pd.DataFrame([{'ID': key, 'Name': technique_catalog.get(key, name)} for key, name in FALLBACK_TECHNIQUES.items()]))

Loaded 706 ATT&CK techniques from official MITRE STIX data.
Catalog: Live MITRE Enterprise ATT&CK STIX dataset


,ID,Name
0,T1059.001,PowerShell
1,T1003.001,LSASS Memory
2,T1110,Brute Force
3,T1566.001,Spearphishing Attachment
4,T1566.002,Spearphishing Link
5,T1071.001,Web Protocols
6,T1021.001,Remote Desktop Protocol
7,T1136.001,Local Account
8,T1098,Account Manipulation
9,T1059.004,Unix Shell


## 3. Create synthetic SIEM logs

These entries are invented for learning. They represent common security-event patterns; they are **not evidence of a real attack**.

In [3]:
logs = [
    {'event_id':'EVT001','timestamp':'2026-10-30 10:00:00','source':'Endpoint','host':'ws-01','user':'alice','event':'PowerShell Execution','details':'Suspicious encoded command launched via PowerShell -enc'},
    {'event_id':'EVT002','timestamp':'2026-10-30 10:02:00','source':'EDR','host':'ws-02','user':'system','event':'Process Access','details':'Unknown process attempted LSASS memory access'},
    {'event_id':'EVT003','timestamp':'2026-10-30 10:04:00','source':'Authentication','host':'auth-01','user':'bob','event':'Failed Logins','details':'15 failed login attempts in 2 minutes'},
    {'event_id':'EVT004','timestamp':'2026-10-30 10:06:00','source':'Email Gateway','host':'mail-01','user':'charlie','event':'Suspicious Email','details':'Macro-enabled invoice attachment from unknown sender'},
    {'event_id':'EVT005','timestamp':'2026-10-30 10:08:00','source':'Web Proxy','host':'proxy-01','user':'unknown','event':'HTTP Beacon','details':'Periodic HTTP callbacks to an unfamiliar external host'},
    {'event_id':'EVT006','timestamp':'2026-10-30 10:10:00','source':'Windows Event Log','host':'finance-01','user':'guest','event':'RDP Access','details':'Unexpected RDP login to finance server after hours'},
    {'event_id':'EVT007','timestamp':'2026-10-30 10:12:00','source':'Endpoint','host':'ws-03','user':'administrator','event':'Account Provisioning','details':'New local admin user created on a workstation'},
    {'event_id':'EVT008','timestamp':'2026-10-30 10:14:00','source':'Identity','host':'ad-01','user':'operator','event':'Group Membership Change','details':'User added to Administrators group unexpectedly'},
    {'event_id':'EVT009','timestamp':'2026-10-30 10:16:00','source':'Linux Audit','host':'web-01','user':'service','event':'Shell Execution','details':'Unexpected shell execution from a service account'},
    {'event_id':'EVT010','timestamp':'2026-10-30 10:18:00','source':'Email Gateway','host':'mail-01','user':'dana','event':'Phishing Link','details':'Suspicious phishing link in an unexpected email'},
    {'event_id':'EVT011','timestamp':'2026-10-30 10:20:00','source':'Authentication','host':'auth-01','user':'alice','event':'Normal Sign-In','details':'Successful scheduled login from a trusted workstation'},
    {'event_id':'EVT012','timestamp':'2026-10-30 10:22:00','source':'Backup','host':'backup-01','user':'backup-service','event':'Backup Success','details':'Automated scheduled backup completed successfully'},
]
log_df = pd.DataFrame(logs)
log_df.to_csv(PROJECT / 'siem_logs.csv', index=False)
print(f'{len(log_df)} synthetic SIEM events created.')
display(log_df)

12 synthetic SIEM events created.


,event_id,timestamp,source,host,user,event,details
0,EVT001,2026-10-30 10:00:00,Endpoint,ws-01,alice,PowerShell Execution,Suspicious encoded command launched via PowerS...
1,EVT002,2026-10-30 10:02:00,EDR,ws-02,system,Process Access,Unknown process attempted LSASS memory access
2,EVT003,2026-10-30 10:04:00,Authentication,auth-01,bob,Failed Logins,15 failed login attempts in 2 minutes
3,EVT004,2026-10-30 10:06:00,Email Gateway,mail-01,charlie,Suspicious Email,Macro-enabled invoice attachment from unknown ...
4,EVT005,2026-10-30 10:08:00,Web Proxy,proxy-01,unknown,HTTP Beacon,Periodic HTTP callbacks to an unfamiliar exter...
5,EVT006,2026-10-30 10:10:00,Windows Event Log,finance-01,guest,RDP Access,Unexpected RDP login to finance server after h...
6,EVT007,2026-10-30 10:12:00,Endpoint,ws-03,administrator,Account Provisioning,New local admin user created on a workstation
7,EVT008,2026-10-30 10:14:00,Identity,ad-01,operator,Group Membership Change,User added to Administrators group unexpectedly
8,EVT009,2026-10-30 10:16:00,Linux Audit,web-01,service,Shell Execution,Unexpected shell execution from a service account
9,EVT010,2026-10-30 10:18:00,Email Gateway,mail-01,dana,Phishing Link,Suspicious phishing link in an unexpected email


## 4. Detect suspicious indicators and map them to ATT&CK

Each explicit rule matches selected text patterns and assigns a **candidate** ATT&CK technique and a demo severity. This is a heuristic matching system: confirm suspicious alerts with further investigation before treating them as real incidents.

In [4]:
RULES = [
    {'name':'Encoded PowerShell command','pattern':r'encoded command|powershell\s+-enc\b|frombase64string','technique_id':'T1059.001','severity':'High'},
    {'name':'Unusual LSASS access','pattern':r'\blsass\b.*\b(access|dump|read)\b|\b(dump|read)\b.*\blsass\b','technique_id':'T1003.001','severity':'Critical'},
    {'name':'Repeated authentication failures','pattern':r'\b\d+ failed login attempts\b|brute[ -]force|password spray','technique_id':'T1110','severity':'Medium'},
    {'name':'Suspicious email attachment','pattern':r'macro-enabled.*attachment|phishing attachment','technique_id':'T1566.001','severity':'High'},
    {'name':'Suspicious email link','pattern':r'phishing link|suspicious.*email link','technique_id':'T1566.002','severity':'High'},
    {'name':'Unusual HTTP beaconing','pattern':r'periodic http callbacks|http beacon|http beaconing','technique_id':'T1071.001','severity':'Medium'},
    {'name':'Unexpected RDP activity','pattern':r'unexpected rdp|suspicious rdp|rdp from untrusted','technique_id':'T1021.001','severity':'Medium'},
    {'name':'Unexpected local account creation','pattern':r'new local admin user|new local account','technique_id':'T1136.001','severity':'Medium'},
    {'name':'Unexpected privilege change','pattern':r'added to administrators group|privilege added','technique_id':'T1098','severity':'High'},
    {'name':'Unexpected Unix shell use','pattern':r'unexpected shell execution|bash script from /tmp','technique_id':'T1059.004','severity':'Medium'},
]

findings = []
for _, log in log_df.iterrows():
    content = f"{log['event']} | {log['details']}"
    for rule in RULES:
        if re.search(rule['pattern'], content, flags=re.IGNORECASE):
            tid = rule['technique_id']
            findings.append({
                'event_id': log['event_id'], 'timestamp': log['timestamp'],
                'source': log['source'], 'host': log['host'], 'user': log['user'],
                'detection': rule['name'], 'severity': rule['severity'],
                'mitre_id': tid,
                'mitre_technique': technique_catalog.get(tid, FALLBACK_TECHNIQUES[tid]),
                'mitre_url': 'https://attack.mitre.org/techniques/' + tid.replace('.', '/') + '/',
                'evidence': log['details'], 'status':'Candidate for analyst review'
            })

findings_df = pd.DataFrame(findings)
findings_df.to_csv(PROJECT / 'threat_findings.csv', index=False)
pd.DataFrame([{k: v for k, v in rule.items() if k != 'pattern'} for rule in RULES]).to_csv(PROJECT / 'detection_rules.csv', index=False)

print(f'Total events: {len(log_df)} | Matched findings: {len(findings_df)} | Unmatched events: {len(log_df)-len(findings_df.event_id.unique())}')
display(findings_df[['event_id','detection','severity','mitre_id','mitre_technique','status']])

Total events: 12 | Matched findings: 10 | Unmatched events: 2


,event_id,detection,severity,mitre_id,mitre_technique,status
0,EVT001,Encoded PowerShell command,High,T1059.001,PowerShell,Candidate for analyst review
1,EVT002,Unusual LSASS access,Critical,T1003.001,LSASS Memory,Candidate for analyst review
2,EVT003,Repeated authentication failures,Medium,T1110,Brute Force,Candidate for analyst review
3,EVT004,Suspicious email attachment,High,T1566.001,Spearphishing Attachment,Candidate for analyst review
4,EVT005,Unusual HTTP beaconing,Medium,T1071.001,Web Protocols,Candidate for analyst review
5,EVT006,Unexpected RDP activity,Medium,T1021.001,Remote Desktop Protocol,Candidate for analyst review
6,EVT007,Unexpected local account creation,Medium,T1136.001,Local Account,Candidate for analyst review
7,EVT008,Unexpected privilege change,High,T1098,Account Manipulation,Candidate for analyst review
8,EVT009,Unexpected Unix shell use,Medium,T1059.004,Unix Shell,Candidate for analyst review
9,EVT010,Suspicious email link,High,T1566.002,Spearphishing Link,Candidate for analyst review


## 5. Verify tests

We check that expected synthetic indicators are detected and that the two routine events are not flagged. This is a lab check, not an evaluation of real-world detection accuracy.

In [5]:
assert len(log_df) == 12, 'Expected 12 sample events.'
assert len(findings_df) == 10, f'Expected 10 mapped findings, received {len(findings_df)}.'
assert 'T1003.001' in set(findings_df['mitre_id']), 'Missing LSASS technique.'
assert not {'EVT011', 'EVT012'}.intersection(set(findings_df['event_id'])), 'Benign events should not be flagged.'
assert not findings_df['mitre_technique'].isna().any(), 'Missing mapped technique names.'
print('PASS: 12 synthetic logs, 10 rule detections, 2 unflagged routine events, and valid MITRE mappings.')

PASS: 12 synthetic logs, 10 rule detections, 2 unflagged routine events, and valid MITRE mappings.


## 6. Build the interactive dashboard and export reports

The dashboard displays alert severity and mapped technique frequency. Hover over the chart to inspect counts. You can download `soc_dashboard.html` and open it in a browser with an internet connection.

In [6]:
severity_order = ['Critical','High','Medium','Low']
severity_counts = findings_df['severity'].value_counts().reindex(severity_order, fill_value=0).reset_index()
severity_counts.columns = ['Severity','Alerts']

fig1 = px.bar(severity_counts, x='Severity', y='Alerts', color='Severity',
              title='SOC Candidate Alerts by Severity',
              category_orders={'Severity': severity_order}, text='Alerts')
fig1.update_layout(showlegend=False, template='plotly_white')
fig1.show()

technique_counts = findings_df.groupby(['mitre_id','mitre_technique']).size().reset_index(name='Alerts')
technique_counts = technique_counts.sort_values('Alerts', ascending=False)
fig2 = px.bar(technique_counts, x='mitre_id', y='Alerts', hover_data=['mitre_technique'],
              title='Candidate MITRE ATT&CK Mappings', text='Alerts')
fig2.update_layout(template='plotly_white', xaxis_title='Technique ID')
fig2.show()

from plotly.io import to_html
from html import escape
safe_table = findings_df[['event_id','detection','severity','mitre_id','mitre_technique','status']].to_html(index=False, escape=True, classes='findings')
body = f"""<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>SOC MITRE ATT&CK Lab</title>
<style>body{{font-family:Arial,sans-serif;background:#f4f7fa;color:#18263a;max-width:1150px;margin:30px auto;padding:0 18px}}h1{{font-size:28px}}.summary{{display:flex;gap:14px;flex-wrap:wrap}}.metric{{background:#fff;border:1px solid #e4e9f0;border-radius:10px;padding:20px;min-width:170px}}.metric strong{{display:block;font-size:27px}}section{{background:white;border-radius:12px;padding:14px;margin:18px 0;overflow:auto}}table{{border-collapse:collapse;width:100%}}th,td{{text-align:left;padding:10px;border-bottom:1px solid #e7edf2;font-size:13px}}th{{background:#eff3f7}}.note{{color:#4d6275;font-size:13px}}</style></head><body>
<h1>SOC Threat Detection &amp; MITRE ATT&amp;CK Mapping</h1><p class="note">Demonstration using 12 simulated SIEM events. Alerts are rule-based candidates, not confirmed compromises.</p>
<div class="summary"><div class="metric">Total Events<strong>{len(log_df)}</strong></div><div class="metric">Candidate Alerts<strong>{len(findings_df)}</strong></div><div class="metric">Routine Events Unflagged<strong>{len(log_df)-findings_df.event_id.nunique()}</strong></div><div class="metric">Unique ATT&amp;CK IDs<strong>{findings_df.mitre_id.nunique()}</strong></div></div>
<section>{to_html(fig1, include_plotlyjs='cdn', full_html=False)}</section>
<section>{to_html(fig2, include_plotlyjs=False, full_html=False)}</section>
<section><h2>Mapped findings</h2>{safe_table}</section>
<p class="note">ATT&amp;CK data: {escape(data_source)}. © The MITRE Corporation. MITRE ATT&amp;CK is a trademark of The MITRE Corporation.</p></body></html>"""
(PROJECT / 'soc_dashboard.html').write_text(body, encoding='utf-8')

severity_summary = severity_counts[severity_counts.Alerts > 0]
report_lines = [
    'SOC THREAT DETECTION & MITRE ATT&CK REPORT',
    '==========================================',
    'Dataset: SYNTHETIC SIEM events (not live production data)',
    f'ATT&CK catalog source: {data_source}',
    f'Total input events: {len(log_df)}',
    f'Candidate alerts: {len(findings_df)}',
    f'Unflagged routine events: {len(log_df) - findings_df.event_id.nunique()}',
    f'Unique ATT&CK techniques mapped: {findings_df.mitre_id.nunique()}',
    '', 'SEVERITY COUNTS',
]
report_lines.extend(f"{row.Severity}: {row.Alerts}" for row in severity_summary.itertuples())
report_lines.extend(['', 'DETECTION FINDINGS'])
report_lines.extend(f"{row.event_id}: {row.detection} -> {row.mitre_id} ({row.severity})" for row in findings_df.itertuples())
report_lines += ['', 'ASSESSMENT: All matches are candidate indicators requiring analyst review.',
                 'LIMITATIONS: Synthetic data, basic keyword/regex rules, and heuristic severity. No AI model trained.',
                 'MITRE ATT&CK: https://attack.mitre.org/']
(PROJECT / 'security_report.txt').write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
print('Saved dashboard and security report to', PROJECT)

Saved dashboard and security report to /content/SOC_MITRE_Project


## 7. Download outputs for GitHub

When this cell finishes, your browser will download **`SOC_MITRE_Results.zip`**. Extract it on your computer. It contains the CSV files, report, and interactive dashboard. Separately save this Colab notebook using **File → Download → Download .ipynb** so recruiters can inspect the actual source code.

In [7]:
import zipfile

readme = """# SOC Threat Detection and MITRE ATT&CK Mapping

A Python / Google Colab security operations lab that uses **synthetic SIEM logs**,
rule-based candidate detection, and MITRE ATT&CK technique mapping.

## Project features
- 12 simulated security events, including two routine activities
- 10 explicit regex-based detection rules
- Candidate severity scoring (Critical / High / Medium)
- MITRE Enterprise ATT&CK technique IDs and names
- An interactive Plotly SOC dashboard and exportable CSV findings
- Basic validation assertions

## How to run
1. Open `SOC_Threat_Detection_MITRE_ATTACK.ipynb` in Google Colab.
2. Choose Runtime > Run all.
3. The notebook will try to fetch MITRE's official Enterprise STIX dataset.
   If the download fails, it uses a limited built-in reference mapping.
4. Download `SOC_MITRE_Results.zip` and inspect `security_report.txt`.
5. Open `soc_dashboard.html` in your browser (chart interactivity needs internet).

## Output files
- `siem_logs.csv`: fictional input events
- `detection_rules.csv`: rules used in the lab
- `threat_findings.csv`: mapped candidate alerts
- `security_report.txt`: text summary and limitations
- `soc_dashboard.html`: interactive charts and findings table

## Accuracy and safety
This is a **rule-based prototype, not a trained AI model or live SIEM**.
A rule match is not proof of malicious behavior. Severity scores are illustrative.
The data is synthetic; no real accounts, network captures, or credentials are used.

## Attribution
MITRE ATT&CK: https://attack.mitre.org/
STIX source: https://github.com/mitre/cti
ATT&CK is a trademark of The MITRE Corporation.
"""
(PROJECT / 'README.md').write_text(readme, encoding='utf-8')

archive = Path('/content/SOC_MITRE_Results.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(PROJECT.iterdir()):
        if path.is_file():
            zf.write(path, arcname=path.name)
print('Created', archive)
print('Files:', [path.name for path in sorted(PROJECT.iterdir())])

try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Not running in Colab. The ZIP file is saved locally.')

Created /content/SOC_MITRE_Results.zip
Files: ['README.md', 'detection_rules.csv', 'security_report.txt', 'siem_logs.csv', 'soc_dashboard.html', 'threat_findings.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>